# Explorer 04 - Labels and the majors

**Purpose.** Rebuild the Universal, Sony, and Warner label trees, measure major versus independent share, and profile who dominates each label.

**Feeds main report section:** 5.3 RQ3 - Label power

**Modules used:** `vinyl.labels`, `vinyl.viz`

**Tables read:** `labels`, `release_labels`, `release_artists`, `release_credits`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-04-labels")
from vinyl import labels as label_power, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
labels = ingest.read_table(spark, snapshot, "labels")
release_labels = ingest.read_table(spark, snapshot, "release_labels")
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_credits = ingest.read_table(spark, snapshot, "release_credits")

## 1. Find the root labels of the three majors

`config.MAJOR_ROOTS` holds placeholder names. Confirm them here first.

In [ ]:
labels.where(F.col("label_name").rlike("(?i)^(universal music|sony music|warner music)")).show(50, truncate=False)

## 2. Resolve every label to its root

In [ ]:
roots = label_power.resolve_label_roots(labels, max_depth=12).cache()
roots.groupBy("reached_root").count().show()          # False rows: raise max_depth or inspect
roots.groupBy("depth").count().orderBy("depth").show()

In [ ]:
tagged = label_power.tag_major_labels(roots)
tagged.groupBy("major").count().show()

## 3. Major versus independent share by year

In [ ]:
major_share = label_power.major_share_by_year(release_labels, tagged).toPandas()
viz.plot_lines(major_share, "year", "share", "major", title="TODO: major vs independent share", percent=True)

## 4. Artist concentration per label

In [ ]:
real_artists = release_artists.where(F.col("artist_id") != 194)  # drop "Various"
concentration = label_power.artist_concentration(release_labels, real_artists, min_releases=50)
concentration.join(tagged.select("label_id", "major"), "label_id").groupBy("major").agg(
    F.count("*").alias("n_labels"), F.avg("hhi").alias("mean_hhi"), F.avg("top_artist_share").alias("mean_top_share")
).show()

## 5. Top producers and writers per label

In [ ]:
producers = label_power.top_contributors(release_labels, release_credits, config.PRODUCTION_ROLES, top_n=10)
writers = label_power.top_contributors(release_labels, release_credits, config.WRITING_ROLES, top_n=10)
producers.orderBy("label_id", "rank").show(20, truncate=False)

In [ ]:
# TODO: most common credit roles - update config.PRODUCTION_ROLES / WRITING_ROLES to match.
release_credits.groupBy("role").count().orderBy(F.desc("count")).show(40, truncate=False)

In [ ]:
# TODO: roll concentration up to the whole major tree, not only single labels.
# TODO: how many releases sit on labels with no parent information at all?

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] Exact root label names or ids for the three majors.
- [ ] `max_depth` needed, and how many labels never reach a root.
- [ ] Minimum releases per label for the concentration measures.
- [ ] Final role lists for producers/engineers and writers.